# plan of action

# using optuna in this code we will try to find best hyperparams using optuna

# number of hidden layers
# neurons per layer
# Number of epochs
# optimizer
# learning rate
# batch size
# dropout rate
# weight decay(lambda)

## optuna code sequence

# objective function
# search space
# mode init
# param init
# training loop
# evaluation loop

In [ ]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
print(device)

cuda


In [ ]:
df = pd.read_csv("/fashion-mnist_train.csv")

In [ ]:
X = df.iloc[:, 1:].values
y = df.iloc[: , 0].values

In [ ]:
X_train , x_test , y_train, y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

In [ ]:
X_train = X_train/255.0
x_test = x_test/255.0

In [ ]:
class DeclareDataSet(Dataset):
    def __init__(self, n_features , labels):
      self.n_features = torch.tensor(n_features , dtype=torch.float32)
      self.labels = torch.tensor(labels , dtype=torch.long)

    def __len__(self):
      return len(self.n_features)

    def __getitem__(self, index):
      return self.n_features[index] , self.labels[index]


In [ ]:
train_dataset = DeclareDataSet(X_train , y_train)
test_dataset = DeclareDataSet(x_test , y_test)

In [ ]:
# train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
class neural_net_optuna(nn.Module):

  def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate):

    super().__init__()

    layers = []

    for i in range(num_hidden_layers):

      layers.append(nn.Linear(input_dim, neurons_per_layer))
      layers.append(nn.BatchNorm1d(neurons_per_layer))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate))
      input_dim = neurons_per_layer

    layers.append(nn.Linear(neurons_per_layer, output_dim))

    self.model = nn.Sequential(*layers) # * unpacks the list as sequential needs unpacked layers


  def forward(self, X):
    return self.model(X)

## here we will use optuna

In [ ]:
def objective(trail):

  # next hyperparam values from defined search space

  num_hidden_layers = trail.suggest_int("num_hidden_layers", 1 , 5) # between 1 to 5 hidden layers select any of one the best
  neurons_per_layer = trail.suggest_int("neurons_per_layer", 8, 128, step=8) # jump of 8 between 8 to 128 picks the best neurons number for layer between 8&128 jumping of 8
  epoches = trail.suggest_int("epoches", 10, 50, step=10)
  learning_rate = trail.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
  dropout_rate = trail.suggest_float("dropout_rate", 0.1, 0.5, step=0.1)
  batch_size = trail.suggest_categorical("batch_size", [16, 32, 64, 128]) # as batch size was defined outside of obj function so we have to bring here
  optimizer_name = trail.suggest_categorical("optimizer_name", ['Adam', 'SGD', 'RMSprop'])
  weight_decay = trail.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

  train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

  # model init

  input_dim = 784
  output_dim = 10

  model = neural_net_optuna(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate)
  model.to(device)

  # param init
  # learning_rate = 0.1
  # epoches = 50

  # optimizer selection
  lossed = nn.CrossEntropyLoss()
  # optimizer = optim.SGD(model.parameters() , lr=learning_rate, weight_decay=1e-4)

  if optimizer_name == 'Adam':
    optimizer = optim.Adam(model.parameters() , lr=learning_rate, weight_decay=weight_decay)

  elif optimizer_name == 'SGD':
    optimizer = optim.SGD(model.parameters() , lr=learning_rate, weight_decay=weight_decay)

  else:
    optimizer = optim.RMSprop(model.parameters() , lr=learning_rate, weight_decay=weight_decay)


  # trianing loop

  for epoch in range(epoches):

  # total_loss_epoch = 0

    for batch_features, batch_labels in train_dataloader:


      # move to GPU
      batch_features , batch_labels = batch_features.to(device) , batch_labels.to(device)

      # call forward pass
      outputs = model(batch_features)

      # calc loss
      loss = lossed(outputs , batch_labels)


      # clear grads
      optimizer.zero_grad()

      # backpropagation
      loss.backward()

      # gradients update
      optimizer.step()

      #total_loss_epoch = total_loss_epoch + loss.item()


    #average = total_loss_epoch/len(train_dataloader)
    #print(f"epoch {epoch + 1 } , average loss of epoch : {average}")

  # eval loop

  model.eval()

  total = 0
  correct = 0

  with torch.no_grad():

    for batch_features , batch_labels in test_dataloader:

      batch_features , batch_labels = batch_features.to(device) , batch_labels.to(device)

      # forward pass
      outputs = model(batch_features)

      _, predicted = torch.max(outputs , 1)

      total = total + batch_labels.shape[0]

      correct = correct + (predicted == batch_labels).sum().item()

    # print(f"model accuracy : { correct/total * 100 } %")

    accuracy = correct/total

  return accuracy

In [ ]:
!pip install optuna

## What is a Study?

A Study is the object that manages the entire hyperparameter optimization process.

Think of it as a manager whose job is to:

Keep track of every trial
Remember which hyperparameters were tried
Record the performance of each trial
Decide what hyperparameters to try next
Store the best result

You can think of it like this:

Study
│
├── Trial 1
├── Trial 2
├── Trial 3
├── ...
└── Best Trial

So this line creates that manager.

In [ ]:
import optuna

study = optuna.create_study(direction='maximize')# if objective function returns accuracy, higher is better, so you use: maximize, if we were finding loss then use minimize

[I 2026-07-26 07:58:38,494] A new study created in memory with name: no-name-990803b5-5b38-4ba6-8f6b-c505f844ebda


## For example:

### Trial 1
Layers = 2
Neurons = 32
Accuracy = 91%


### Trial 2
Layers = 5
Neurons = 64
Accuracy = 94%

... up to 10 trails


In [ ]:
study.optimize(objective, n_trials=10)

[I 2026-07-26 07:59:11,716] Trial 0 finished with value: 0.8765833333333334 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 80, 'epoches': 20, 'learning_rate': 6.705939873475804e-05, 'dropout_rate': 0.2, 'batch_size': 128, 'optimizer_name': 'RMSprop', 'weight_decay': 1.40427163115206e-05}. Best is trial 0 with value: 0.8765833333333334.
[I 2026-07-26 08:06:04,954] Trial 1 finished with value: 0.8714166666666666 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 56, 'epoches': 50, 'learning_rate': 1.510971279214824e-05, 'dropout_rate': 0.2, 'batch_size': 16, 'optimizer_name': 'RMSprop', 'weight_decay': 0.00025257162974727743}. Best is trial 0 with value: 0.8765833333333334.
[I 2026-07-26 08:07:36,584] Trial 2 finished with value: 0.8465833333333334 and parameters: {'num_hidden_layers': 2, 'neurons_per_layer': 40, 'epoches': 50, 'learning_rate': 0.0008403504456800672, 'dropout_rate': 0.4, 'batch_size': 64, 'optimizer_name': 'SGD', 'weight_decay': 2.66076842322928

In [ ]:
study.best_value

0.8845

In [ ]:
study.best_params

{'num_hidden_layers': 2,
 'neurons_per_layer': 64,
 'epoches': 20,
 'learning_rate': 0.002756551741474784,
 'dropout_rate': 0.1,
 'batch_size': 64,
 'optimizer_name': 'Adam',
 'weight_decay': 6.371762901642232e-05}

In [ ]:
# class neural_network(nn.Module):
#   def __init__(self , n_features):
#     super().__init__()

#     self.module = nn.Sequential(
#         nn.Linear(n_features, 128),
#         nn.BatchNorm1d(128),   # normalization apply: before activation and after the hidden layer
#         nn.ReLU(),
#         nn.Dropout(p=0.3),
#         nn.Linear(128, 64),
#         nn.BatchNorm1d(64),
#         nn.ReLU(),
#         nn.Dropout(p=0.3),
#         nn.Linear(64, 10)

#     )

#   def forward(self, X):
#     return self.module(X)



In [ ]:
# model = neural_network(X_train.shape[1])

In [ ]:
# model.to(device)

In [ ]:
# learning_rate = 0.1
# epoches = 30

In [ ]:
# lossed = nn.CrossEntropyLoss()

In [ ]:
# optimizer = optim.SGD(model.parameters() , lr=learning_rate, weight_decay=1e-4)

In [ ]:
# for epoch in range(epoches):

#   total_loss_epoch = 0

#   for batch_features , batch_labels in train_dataloader:


#     # move to GPU
#     batch_features , batch_labels = batch_features.to(device) , batch_labels.to(device)

#     # call forward pass
#     outputs = model(batch_features)

#     # calc loss

#     loss = lossed(outputs , batch_labels)


#     # clear grads

#     optimizer.zero_grad()

#     # backpropagation

#     loss.backward()

#     # gradients update

#     optimizer.step()

#     total_loss_epoch = total_loss_epoch + loss.item()


#   average = total_loss_epoch/len(train_dataloader)
#   print(f"epoch {epoch + 1 } , average loss of epoch : {average}")

In [ ]:
# model.eval()

In [ ]:
# total = 0
# correct = 0

In [ ]:
# with torch.no_grad():

#   for batch_features , batch_labels in test_dataloader:

#     batch_features , batch_labels = batch_features.to(device) , batch_labels.to(device)
#     # forward pass
#     outputs = model(batch_features)

#     _, predicted = torch.max(outputs , 1)

#     total = total + batch_labels.shape[0]

#     correct = correct + (predicted == batch_labels).sum().item()

#   print(f"model accuracy : { correct/total * 100 } %")


In [ ]:
# correct = 0
# total = 0

# with torch.no_grad():

#   for batch_features , batch_labels in train_dataloader:

#     batch_features , batch_labels = batch_features.to(device) , batch_labels.to(device)
#     # forward pass
#     outputs = model(batch_features)

#     _, predicted = torch.max(outputs , 1)

#     total = total + batch_labels.shape[0]

#     correct = correct + (predicted == batch_labels).sum().item()

#   print(f"model accuracy : { correct/total * 100 } %")

In [ ]:
!pip install mlflow dagshub --quiet



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.3/273.3 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121